# Week 2 Day 2 assignment, task 1: a word lookup in place of Qdrant

The memory agent from the lab's Experiment 2, with `qdrant-find` replaced by a word lookup over `data/knowledge.json`. The lab's code is in `week2day2_common.py`. Run the lab through Part 4 first, for its Qdrant memory and its Langfuse dataset. Runs are saved in `results/`, so running again makes no model calls.

In [ ]:
import re

from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware
from langchain_core.tools import tool
from week2day2_common import MEMORY_LINE, by_claim, calculate_payout, compare, decide, knowledge, luna, make_system_prompt, memory_search_tools, record_decision, run_experiment

## The word lookup

Each fact scores one point for each query word it contains, ignoring case, common words and a plural s. Up to 10 facts come back word for word, best first, so `found_fact` still works.

In [ ]:
STOP_WORDS = set("a an and any are as at be before by for from had has have in into is it its of on or that the this to was were what which who will with".split())


def words(text):
    """The words in a text, lowercased, without common words, and with a plural s taken off."""
    found = re.findall(r"[a-z0-9]+", text.lower())
    return {word[:-1] if len(word) > 3 and word.endswith("s") else word for word in found if len(word) > 1 and word not in STOP_WORDS}


FACT_WORDS = [words(entry["fact"]) for entry in knowledge]


def word_lookup(query, limit=10):
    """The facts that share the most words with the query, best first."""
    query_words = words(query)
    scores = [(len(query_words & fact_words), index) for index, fact_words in enumerate(FACT_WORDS)]
    scores = sorted([score for score in scores if score[0] > 0], key=lambda score: -score[0])
    return [knowledge[index]["fact"] for _, index in scores[:limit]]


@tool(parse_docstring=True)
def lookup_facts(query: str) -> str:
    """Search your memory for facts that contain the words in the query, best match first.

    Args:
        query: The words to look for, such as the name of a product, an event or a place
    """
    facts = word_lookup(query)
    if not facts:
        return f"Nothing in your memory contains the words in '{query}'"
    return f"Results for the query '{query}'\n" + "\n".join(f"- {fact}" for fact in facts)


print(word_lookup("Halden Arc")[0])
print(word_lookup("a gadget for watching films on a wall"))

For the lab's example of searching by meaning, Qdrant puts the Halden Arc first. The word lookup finds only the Sentinel Wall dive site.

## Experiment 2 by meaning and by words, 3 runs each

Only the search tool differs. `decide` picks out the search results by the tool's name.

In [ ]:
memory_search = await memory_search_tools()
qdrant_agent = create_agent(model=luna, tools=[record_decision, calculate_payout, *memory_search], system_prompt=make_system_prompt(MEMORY_LINE), middleware=[ToolRetryMiddleware(max_retries=3)])
word_agent = create_agent(model=luna, tools=[record_decision, calculate_payout, lookup_facts], system_prompt=make_system_prompt(MEMORY_LINE))


async def qdrant_task(*, item, **kwargs):
    return await decide(qdrant_agent, item.input, search_tools=["qdrant-find"])


async def word_task(*, item, **kwargs):
    return await decide(word_agent, item.input, search_tools=["lookup_facts"])


tables = {
    "Search by meaning": run_experiment("Payout tool and memory", qdrant_task, "qdrant"),
    "Search by words": run_experiment("Payout tool and word lookup", word_task, "word_lookup"),
}
compare(tables)

In [ ]:
by_claim(tables)

## Conclusion

**Searching by words scores the same as searching by meaning on these claims, and takes half the time.**

| | By meaning (Qdrant) | By words |
|---|---:|---:|
| Right, over 3 runs | 70 of 75 (23, 24, 23) | 70 of 75 (25, 22, 23) |
| Found the fact the claim needs | 100% | 100% |
| Cost of mistakes, per run | $590 | $636 |
| Model cost, per run | $0.012 | $0.011 |
| Seconds per claim | 14.7 | 7.7 |

- Retrieval isn't what goes wrong. Both searches found the needed fact on every claim in every run, and every mistake came after that: a misread storm or strike date, a referral where the wording settled it, or a wrong payout.
- The claims name rare proper nouns that are also in their facts (Halden Arc, Tallow Gorge, Marisol), which is where a word lookup is strongest. It fails when the words differ, as the projector example shows.
- The speed comes from the plumbing. Each `qdrant-find` call starts the MCP server with `uvx` and loads the embedding model; the lookup is a function over 33 facts in memory.
- The variation between runs is larger than the difference between the two searches. The lab's single Qdrant run got 25 of 25; these three got 23, 24 and 23.

For 33 short facts full of names I would use the word lookup: as accurate, faster and nothing to install. Search by meaning earns its place when the knowledge base grows and its wording drifts from the claims'.